# Voir où la tête d'attention écoute : jetons et poids sur le spectrogramme

La tête attentive (`blanci/heads/attentive.py`) donne un poids à chaque jeton d'une fenêtre. Avec perch_v2, les 64 jetons forment une grille de 16 pas de temps × 4 bandes de fréquence : chaque jeton couvre un rectangle du spectrogramme. Ce notebook superpose les poids au spectrogramme, pour juger **à l'œil** si la tête écoute la note d'A. blanci (4,4–5,5 kHz) ou le fond, et en tire une mesure : la **part d'attention dans la bande de la note**, comparée à une attention uniforme.

- Prérequis : un stock d'embeddings (`blanci embed`) et les jetons des fenêtres du benchmark (`blanci tokens --encoder perch_v2`).
- **Lecture seule** : la base est ouverte en lecture, l'audio est lu, jamais écrit.
- **Ne pas committer les sorties** (lecteurs audio) : *Clear All Outputs* avant un commit.
- La géométrie des jetons (ordre, sens des fréquences, bornes) est une hypothèse : la section 4 la contrôle avec des sons purs. À faire une fois par encodeur avant de croire la carte.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupKFold

from blanci.core.config import config_path, load_config
from blanci.evaluation.head_benchmark import benchmark_data
from blanci.exploration import explore as ex
from blanci.exploration import explore_plots as ep
from blanci.exploration.attention_map import (
    GRIDS,
    attention_grid,
    band_share,
    peak_ratio,
    plot_attention,
    uniform_band_share,
)
from blanci.heads.attentive import AttentiveHead, fit_attentive
from blanci.heads.pooling import flat_tokens
from blanci.inputs.dataset import recordings_table

# Racine du dépôt : les chemins de la config y sont relatifs (le notebook tourne dans notebooks/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
pd.set_option("display.width", 200)

## Réglages

In [ ]:
CONFIG = ROOT / "config" / "local.yaml"
STOCK = "perch_v2-bacpipe1.3.5"  # stock d'embeddings et de jetons
GRILLE = GRIDS["perch_v2"]  # géométrie des jetons (section 4 pour la contrôler)
TETE = None  # dossier d'une tête attentive sauvegardée ; None : entraînée ici, hors pli
PLIS = 5  # plis groupés par point d'écoute pour les poids hors pli
OPTIONS = {}  # options de fit_attentive, ex. {"optimizer": "adamw", "entropy": 0.01}
BANDE = (4400, 5500)  # bande de la note d'A. blanci (Hz)
N_EXEMPLES = 4  # fenêtres montrées par catégorie

cfg = load_config(CONFIG if CONFIG.exists() else None)
con = ex.open_readonly(config_path(cfg, "db"))
data, X, tokens = benchmark_data(con, cfg, STOCK)
if tokens is None:
    raise RuntimeError(f"pas de jetons pour {STOCK} : lancer `blanci tokens --encoder perch_v2`")
tokens = flat_tokens(tokens)
y = data["y"].to_numpy()
window_s = float(data["dur_s"].iloc[0])
data = data.merge(recordings_table(con)[["recording_id", "path"]], on="recording_id", how="left")
print(
    f"{len(data)} fenêtres ({int(y.sum())} positives), jetons {tokens.shape}, fenêtre {window_s} s"
)

## 1. Poids d'attention, hors pli

Les poids d'une fenêtre viennent d'une tête **qui ne l'a pas vue** : entraînée sur les autres points d'écoute (plis groupés par point). Sinon la carte flatterait la tête sur ses propres exemples. Avec `TETE`, on lit une tête sauvegardée telle quelle (à réserver à des fenêtres qu'elle n'a pas vues).

In [ ]:
weights = np.zeros(tokens.shape[:2])
scores = np.zeros(len(data))
if TETE:
    head = AttentiveHead.load(Path(TETE))
    weights, scores = head.attention(tokens), head.decision(tokens)
else:
    groups = data["point"].to_numpy()
    for k, (train, test) in enumerate(GroupKFold(n_splits=PLIS).split(tokens, y, groups)):
        head = fit_attentive(tokens[train], y[train], **OPTIONS)
        weights[test] = head.attention(tokens[test])
        scores[test] = head.decision(tokens[test])
        print(f"pli {k + 1}/{PLIS} : {len(test)} fenêtres")
grid_w = attention_grid(weights, GRILLE)
data["score"] = scores

## 2. La mesure : part d'attention dans la bande de la note

- **part bande** : somme des poids des jetons qui recouvrent 4,4–5,5 kHz (au prorata du recouvrement) ;
- **uniforme** : la même part pour une attention égale sur tous les jetons ;
- **× uniforme** : le rapport des deux. Une tête qui a appris la note dépasse nettement 1 sur les positives, et reste proche de 1 (ou en dessous) sur les négatives ;
- **pic** : poids du jeton le plus lourd × nombre de jetons (1 = uniforme, 64 = un seul jeton).

C'est une mesure de plus pour comparer des têtes attentives (réglages, régularisations), à côté de l'AP : deux têtes d'AP égale ne sont pas équivalentes si l'une écoute le fond.

In [ ]:
uniform = uniform_band_share(GRILLE, BANDE)
data["part_bande"] = band_share(grid_w, GRILLE, BANDE)
data["x_uniforme"] = data["part_bande"] / uniform
data["pic"] = peak_ratio(weights)
summary = (
    data.groupby("y")[["part_bande", "x_uniforme", "pic"]]
    .median()
    .rename(index={0: "négatives", 1: "positives"})
)
print(f"attention uniforme : {uniform:.3f} dans la bande {BANDE[0]}–{BANDE[1]} Hz")
display(summary.round(3))
if "quality" in data:
    display(data[data["y"] == 1].groupby("quality", dropna=False)["x_uniforme"].describe().round(2))

fig, ax = plt.subplots(figsize=(7, 3))
for label, color in [(0, "#A7BFB2"), (1, "#E9B44C")]:
    ax.hist(
        data.loc[data["y"] == label, "x_uniforme"],
        bins=40,
        alpha=0.8,
        color=color,
        label="positives" if label else "négatives",
    )
ax.axvline(1, color="black", lw=0.8, ls="--")
ax.set_xlabel("part d'attention dans la bande / part uniforme")
ax.legend(frameon=False)
plt.show()

## 3. Les cartes

Trois catégories : les positives les mieux notées, les positives où l'attention est la plus **hors** de la bande (la tête a peut-être appris un raccourci : fond, heure, autre espèce), et les négatives les mieux notées (faux amis probables). Opacité des rectangles ∝ poids ; pointillés : bande de la note.

In [ ]:
def show(rows, title):
    rows = list(rows.itertuples())
    fig, axes = plt.subplots(1, len(rows), figsize=(4.2 * len(rows), 3.2), squeeze=False)
    for ax, row in zip(axes[0], rows, strict=True):
        wav, sr = ex.read_segment(cfg, row.path, row.offset_s, window_s)
        ep.show_spectrogram(ax, wav, sr, band_hz=BANDE, fmax=12_000)
        plot_attention(ax, grid_w[row.Index], GRILLE, window_s, fmax_hz=12_000)
        ax.set_title(f"score {row.score:.2f} · ×{row.x_uniforme:.1f} unif.", fontsize=9)
        ax.set_xlabel("s")
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()
    for row in rows:
        wav, sr = ex.read_segment(cfg, row.path, row.offset_s, window_s)
        display(ep.listen(wav, sr))


positives = data[data["y"] == 1]
show(positives.nlargest(N_EXEMPLES, "score"), "Positives les mieux notées")
show(positives.nsmallest(N_EXEMPLES, "x_uniforme"), "Positives où l'attention sort de la bande")
show(data[data["y"] == 0].nlargest(N_EXEMPLES, "score"), "Négatives les mieux notées")

## 4. Contrôle de la géométrie par un son pur

La carte suppose un ordre des jetons (temps puis fréquence), une bande 0 en bas du spectre et des bandes égales en mel entre `fmin` et `fmax`. On le vérifie : un son pur à f Hz dans du bruit doit changer surtout les jetons de la bande qui contient f. Si la bande la plus touchée ne monte pas avec la fréquence, corriger `GRILLE` (`low_first=False`, bornes, `order`) avant de lire les cartes. Encode à la volée (une minute de chargement au premier appel).

In [ ]:
from blanci.embedding.encoders import get_encoder

encoder = get_encoder("perch_v2", cfg)
sr = encoder.sample_rate
t = np.arange(round(encoder.window_s * sr)) / sr
rng = np.random.default_rng(0)
noise = 0.01 * rng.standard_normal(len(t)).astype(np.float32)
reference = flat_tokens(encoder.embed_tokens(noise[None], sr))[0]
edges = GRILLE.band_edges_hz()
rows = []
for f in (300, 1000, 2500, 4750, 8000, 12000):
    tone = (noise + 0.2 * np.sin(2 * np.pi * f * t)).astype(np.float32)
    toks = flat_tokens(encoder.embed_tokens(tone[None], sr))[0]
    change = np.linalg.norm(toks - reference, axis=1)
    per_band = attention_grid(change[None], GRILLE)[0].mean(axis=0)
    expected = int(np.searchsorted(edges, f) - 1)
    rows.append(
        {
            "f (Hz)": f,
            "bande attendue": expected,
            "bande la plus touchée": int(per_band.argmax()),
            **{f"bande {j}": round(float(v), 2) for j, v in enumerate(per_band)},
        }
    )
display(pd.DataFrame(rows))